In [1]:
! pip install optuna xgboost

  Using cached xgboost-3.0.5-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/400.9 kB ? eta -:--:--
   --- ----------------------------------- 41.0/400.9 kB 991.0 kB/s eta 0:00:01
   ---------------------------------------- 400.9/400.9 kB 5.0 MB/s eta 0:00:00
Using cached xgboost-3.0.5-py3-none-win_amd64.whl (56.8 MB)



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# imports

import pandas as pd
import numpy as np
import mlflow.sklearn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN
import optuna
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

c:\Users\iuri_\OneDrive\Desktop\youtube_sentiment_mlops_pipeline\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
df = pd.read_csv("reddit_preprocessed.csv")
df.head()

,clean_comment,category,words,stop_words,characters,punctuation_chars
0,family mormon never tried explain still stare ...,1,39,13,259,0
1,buddhism much lot compatible christianity espe...,1,196,59,1268,0
2,seriously say thing first get complex explain ...,-1,86,40,459,0
3,learned want teach different focus goal not wr...,0,29,15,167,0
4,benefit may want read living buddha living chr...,1,112,45,690,0


In [6]:
import mlflow

mlflow.set_tracking_uri("http://ec2-18-222-216-138.us-east-2.compute.amazonaws.com:5000")

In [7]:
mlflow.set_experiment("Training and Hyperparameter tuning XGBoost model")

2025/09/24 11:45:00 INFO mlflow.tracking.fluent: Experiment with name 'Training and Hyperparameter tuning XGBoost model' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-078/459817787713190955', creation_time=1758725099179, experiment_id='459817787713190955', last_update_time=1758725099179, lifecycle_stage='active', name='Training and Hyperparameter tuning XGBoost model', tags={}>

In [16]:
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1}) # since xgboost does not allow a category to be -1, we are going to change it using this map function

In [17]:
# best parameters for vectorization and balancing found in previous notebooks
# -----------------
ngram_range = (1,2)
max_feature = 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_feature)

X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=0)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test) # correct way to do it, without data leakage

rus = RandomUnderSampler(random_state=0) 
X_train, y_train = rus.fit_resample(X_train, y_train)
# -----------------

# Function to log results in MLflow
def log_bestmodel_mlflow(model_name, model, X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_undersampling_TFIDF_bigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

def objective_xgboost(trial):

    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=0)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=0)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_bestmodel_mlflow("XGBoost", best_model, X_train, X_test, y_train, y_test)

In [19]:
run_optuna_experiment()

[I 2025-09-24 12:06:18,412] A new study created in memory with name: no-name-027ed509-1d29-430a-b333-7ab457b3e1ce
[I 2025-09-24 12:07:55,530] Trial 0 finished with value: 0.6127175551596546 and parameters: {'n_estimators': 292, 'learning_rate': 0.0018428320018437974, 'max_depth': 8}. Best is trial 0 with value: 0.6127175551596546.
[I 2025-09-24 12:09:21,482] Trial 1 finished with value: 0.6439632725777716 and parameters: {'n_estimators': 216, 'learning_rate': 0.0041187736538512475, 'max_depth': 9}. Best is trial 1 with value: 0.6439632725777716.
[I 2025-09-24 12:10:57,781] Trial 2 finished with value: 0.6967246813759079 and parameters: {'n_estimators': 232, 'learning_rate': 0.013350574505111705, 'max_depth': 10}. Best is trial 2 with value: 0.6967246813759079.
[I 2025-09-24 12:11:33,558] Trial 3 finished with value: 0.7505824311360834 and parameters: {'n_estimators': 135, 'learning_rate': 0.07363151344215164, 'max_depth': 9}. Best is trial 3 with value: 0.7505824311360834.
[I 2025-09-2

🏃 View run XGBoost_undersampling_TFIDF_bigrams at: http://ec2-18-222-216-138.us-east-2.compute.amazonaws.com:5000/#/experiments/459817787713190955/runs/a22fbafe27bc42e187a185251ec6c6e5
🧪 View experiment at: http://ec2-18-222-216-138.us-east-2.compute.amazonaws.com:5000/#/experiments/459817787713190955
